In [1]:
import numpy as np
import torch

#.\venv\Scripts\Activate.ps1

In [2]:
import sys
print(sys.executable)

d:\personalprojects\venv\Scripts\python.exe


In [3]:
class LoRALayer(torch.nn.Module):
    def __init__(self, in_features, out_features, rank, alpha):
        super().__init__()

        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = torch.nn.Parameter(torch.randn(in_features, rank) * std_dev)
        self.B = torch.nn.Parameter(torch.zeros(rank, out_features))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

In [4]:
lora = LoRALayer(26, 26, 4, .1)
lora.A

Parameter containing:
tensor([[-0.7513,  0.3138,  1.3002,  0.2668],
        [-0.0566,  0.6069,  0.1224, -0.5168],
        [ 0.0553, -1.4894, -0.1967,  0.0056],
        [-0.4455, -0.3742,  0.3301,  0.3587],
        [-1.0952,  0.5853, -1.1036, -0.4465],
        [-0.5919, -0.3976,  0.1754, -0.4627],
        [ 0.5370,  0.7872,  0.0742,  0.5211],
        [ 0.0279, -0.7077,  0.3334,  0.3906],
        [ 0.2689,  0.5590, -0.0182, -0.2020],
        [ 0.7358, -0.6656, -0.9298, -0.9120],
        [ 0.8257, -0.1678, -0.0379,  0.1559],
        [-0.0055,  0.9397, -0.5015, -0.6806],
        [ 0.9713, -0.2641, -0.1915, -0.2952],
        [-0.6810,  0.6896, -0.2406, -0.0050],
        [-0.0568,  0.5722,  0.2807, -0.2036],
        [-0.3561,  0.5162,  0.0545,  0.2549],
        [-0.6195, -0.0886, -0.4947, -0.8685],
        [ 0.2097, -0.6449,  0.3211, -0.0811],
        [-0.4892,  0.1692,  0.1876, -0.0288],
        [ 0.0997, -0.2687,  0.5812, -0.2063],
        [ 0.2124,  0.2945,  0.0818, -0.4020],
        [-0.

In [5]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [6]:
linear = torch.nn.Linear(8, 4)
lora_linear = LinearWithLoRA(linear, rank=2, alpha=0.1)

x = torch.randn(3,8)


In [7]:
lora_linear(x)

tensor([[-0.0712,  0.0885,  1.3040,  0.4560],
        [-0.7595, -0.6992,  0.1028, -0.3835],
        [ 0.8810,  0.4017, -0.3330,  0.5918]], grad_fn=<AddBackward0>)

In [8]:
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA GeForce RTX 3060 Ti
8.6 GB


In [9]:
# Source - https://stackoverflow.com/a/79760524
# Posted by Raka Surya Kusuma
# Retrieved 2026-06-05, License - CC BY-SA 4.0
import torch, torchvision, torchaudio
print(torch.__version__, torchvision.__version__, torchaudio.__version__)


2.12.0+cu126 0.27.0+cu126 2.11.0+cu126


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

d:\personalprojects\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 3870.62it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


In [11]:
for param in model.parameters():
    param.requires_grad = False

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

In [12]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (

In [13]:
from functools import partial

# default hyperparameter choices
lora_r = 8
lora_alpha = 16
lora_dropout = 0.05
lora_query = True
lora_key = False
lora_value = True
lora_projection = False
lora_mlp = False
lora_head = False

layers = []

assign_lora = partial(LinearWithLoRA, rank=lora_r, alpha=lora_alpha)

for layer in model.model.layers:
    if lora_query:
        layer.self_attn.q_proj = assign_lora(layer.self_attn.q_proj)
    if lora_key:
        layer.self_attn.k_proj = assign_lora(layer.self_attn.k_proj)
    if lora_value:
        layer.self_attn.v_proj = assign_lora(layer.self_attn.v_proj)
    if lora_projection:
        layer.self_attn.o_proj = assign_lora(layer.self_attn.o_proj)
    if lora_mlp:
        layer.mlp.gate_proj = assign_lora(layer.mlp.gate_proj)
        layer.mlp.up_proj = assign_lora(layer.mlp.up_proj)
        layer.mlp.down_proj = assign_lora(layer.mlp.down_proj)
if lora_head:
    model.lm_head = assign_lora(model.lm_head)

In [14]:
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

In [15]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LinearWithLoRA(
            (linear): Linear(in_features=3072, out_features=3072, bias=False)
            (lora): LoRALayer()
          )
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): LinearWithLoRA(
            (linear): Linear(in_features=3072, out_features=1024, bias=False)
            (lora): LoRALayer()
          )
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Ll

In [16]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/sst2")

In [17]:
ds['train'][:10]

{'idx': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'sentence': ['hide new secretions from the parental units ',
  'contains no wit , only labored gags ',
  'that loves its characters and communicates something rather beautiful about human nature ',
  'remains utterly satisfied to remain the same throughout ',
  'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ',
  "that 's far too tragic to merit such superficial treatment ",
  'demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . ',
  'of saucy ',
  "a depressed fifteen-year-old 's suicidal poetry ",
  "are more deeply thought through than in most ` right-thinking ' films "],
 'label': [0, 0, 1, 0, 0, 0, 1, 1, 0, 1]}

In [18]:
def format_sample(example):
    label = "positive" if example["label"] == 1 else "negative"
    return {
        "text": f"Classify the sentiment as positive or negative.\nText: {example['sentence']}\nSentiment: {label}"
    }

ds = ds.map(format_sample)

In [19]:
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    result = tokenizer(example["text"], truncation=True, padding="max_length", max_length=64)
    result["labels"] = [
        [token if mask == 1 else -100 for token, mask in zip(ids, masks)]
        for ids, masks in zip(result["input_ids"], result["attention_mask"])
    ]
    return result

tokenized = ds.map(tokenize, batched=True)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [20]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="lora-llama-sst2-retry",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=3e-4,
    bf16=True,        # was fp16=True
    logging_steps=50,
    save_steps=500,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"].select(range(5000)),
)

trainer.train()

Step,Training Loss
50,2.236014
100,1.750348
150,1.876825
200,1.827369
250,1.681297
300,1.791623
350,1.686134
400,1.846872
450,1.868342
500,1.717377


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.68s/it]


TrainOutput(global_step=7500, training_loss=1.5432448928833007, metrics={'train_runtime': 867.4068, 'train_samples_per_second': 17.293, 'train_steps_per_second': 8.646, 'total_flos': 1.624919703552e+16, 'train_loss': 1.5432448928833007, 'epoch': 3.0})

In [26]:
for name, param in model.named_parameters():
    if "lora.B" in name:
        print(name, param.abs().max().item())
        break

model.layers.0.self_attn.q_proj.lora.B 0.06683934479951859


In [30]:
def classify_sentiment(text):
    prompt = f"Classify the sentiment as positive or negative.\nText: {text}\nSentiment:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False
        )
    
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return generated.strip()


In [38]:
print(classify_sentiment("why is the train scheduled to arrive at 10:30 AM."))

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


negative
